# SimWorld Nav Experiment

Pre-requisites:
1. UE Editor running with the SimWorld project loaded (PIE will be auto-started via MCP).
2. UnrealCV plugin reachable on `127.0.0.1:9000`.
3. UE editor MCP TCP server reachable on `127.0.0.1:55557` (used to start PIE).
4. `pip install -e C:/Users/28262/Desktop/PlayGorund/task_gen` and the deps in `gym_env/requirements.txt`.
5. Set `ANTHROPIC_API_KEY` (Claude) or `OPENAI_API_KEY` / `GEMINI_API_KEY` / `DASHSCOPE_API_KEY` for the model you want.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-5s %(name)s | %(message)s')

from gym_env import (
    UCVClient, MCPClient, SimWorldNavEnv, EpisodeLogger,
    sample_pointnav_episode, run_episode, make_llm,
)

In [ ]:
# 1. Start PIE via MCP (Editor mode -> PIE), then connect UnrealCV.
mcp = MCPClient(host='127.0.0.1', port=55557, name='env-0-mcp')
mcp.start_pie(wait_seconds=5.0)   # idempotent: no-op if PIE already running

ucv = UCVClient(host='127.0.0.1', port=9000, name='env-0')
ucv.connect()
print('connected, scene actors:', len(ucv.vget_objects()))

In [ ]:
episode = sample_pointnav_episode(
    ucv, seed=42, target_distance_cm=2000, max_steps=30,
)
print(episode.episode_id, '->', episode.goal_position)

In [ ]:
env = SimWorldNavEnv(ucv_client=ucv, mcp_client=mcp)
llm = make_llm('claude')
logger = EpisodeLogger(
    run_name=f'{llm.name}_{episode.episode_id}',
    meta={'model': llm.name, 'model_id': llm.model, 'episode_id': episode.episode_id},
)
metrics = run_episode(env, llm, episode, logger, max_steps=30, vision_history_depth=3)
metrics

In [ ]:
# Quick analysis: load episode jsonl
import json
from pathlib import Path
lines = (logger.dir / 'episode.jsonl').read_text().splitlines()
for ln in lines[:3]:
    print(json.loads(ln)['info'].get('reward'), json.loads(ln)['info'].get('distance_to_goal_cm'))

## Cleanup

In [ ]:
env.close()
logger.close()
ucv.disconnect()